<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_38_fastapi_basics/note_lesson_38_fastapi_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Урок 38 — FastAPI basics: новинний агрегатор, крок 2

В уроці 37 агрегатор `news_hub` навчився **перевіряти** новини: парсер дає сирі рядки, модель `NewsItem` пропускає лише правильні. Сьогодні агрегатор стає **сервісом** — HTTP API на FastAPI: збір новин, фільтри, статистика, опис `/docs`. Сервер запускати не треба — усі запити йдуть через `TestClient`.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | проєкт `news_hub` і `TestClient` | 0.1 | |
| 1 | перші запити: `/health`, збір зі знімка, фільтри, пагінація, статистика | 1.1–1.8 | |
| 2 | `422`: як FastAPI перевіряє параметри й тіло запиту | 2.1–2.4 | |
| 3 | свої ендпоінти: шлях, рядок запиту, тіло, `response_model`, `201`, `404`, порядок маршрутів | 3.1–3.9 | **Задача 1** |
| 4 | `Depends`, `lifespan` і підміна залежності | 4.1–4.4 | **Задача 2** |
| 5 | OpenAPI: `/openapi.json` | 5.1–5.3 | |
| 6 | `async def` чи `def`: виміри | 6.1–6.4 | |
| 7 | знайди помилку: чужий URL і SSRF | 7.1–7.2 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, `@app.get`) лише створює функцію чи маршрут — **запуск завжди в наступній клітинці**. Так можна перезапустити запит, не перевизначаючи ендпоінт.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Задач **три**; перевірка кожної — у клітинці одразу під нею.
- Рефакторинги, архітектура, Postman і виміри на справжньому сервері — у книзі курсу: [Урок 38](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_38/).

---
## 0. Проєкт у ноутбуці

`news_hub` з уроку 37 + `api.py` (FastAPI), `store.py` (сховище в пам'яті) і `scraper.py` (aiohttp). Клітинка нижче:

- у Colab завантажує з GitHub лише папку уроку (локально вона вже поруч з ноутбуком);
- ставить пакети, яких бракує (`fastapi`, `aiohttp`, `bs4`, `httpx`);
- імпортує все, що знадобиться в уроці.

Інтернет потрібен лише Colab-у для завантаження. Усі запити уроку йдуть **без мережі**: новини беремо зі знімка `data/rbc_news_snapshot.json` (168 новин).

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_38_fastapi_basics"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiohttp, bs4, fastapi, httpx  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())

import asyncio
import time
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from typing import Annotated, Literal

import httpx
from fastapi import Depends, FastAPI, HTTPException, Path, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, HttpUrl, ValidationError, field_validator

from news_hub.api import ScrapeRequest, StoreDep, app, get_scrapers
from news_hub.parser import RawNews
from news_hub.scraper import PageResult, ScrapeOutcome

print("FastAPI", fastapi.__version__, "| папка:", os.getcwd())

### Приклад 0.1. `TestClient` — сервер без сервера

`TestClient(app)` надсилає HTTP-запити **прямо в застосунок** — ті самі, що пішли б через мережу до `uvicorn`. `with TestClient(app) as client:` ще й запускає `lifespan`: агрегатор створює порожнє сховище `NewsStore`.

У ноутбуці один клієнт потрібен на весь урок, тож замість `with` викликаємо `client.__enter__()` (а в останній клітинці — `__exit__`).

In [ ]:
client = TestClient(app)
_ = client.__enter__()      # те саме, що with TestClient(app) as client: — але на весь ноутбук

In [ ]:
response = client.get("/health")
print(response.status_code, response.json())

`200 {'status': 'ok'}` — застосунок живий. Далі кожен запит — `client.get(path, params={...})` або `client.post(path, json={...})`, як у `httpx` з уроку 32. Відповідь має `.status_code`, `.json()`, `.headers`.

---
## 1. Перші запити

### Приклад 1.1. Які маршрути є в застосунку

`app.routes` — список усіх маршрутів: метод + шлях → функція. Частину FastAPI додає сам (`/docs`, `/openapi.json`, `/redoc`).

In [ ]:
def show_routes(application: FastAPI) -> None:
    """Надрукувати метод і шлях кожного маршруту застосунку."""
    for route in application.routes:
        methods = ",".join(sorted(getattr(route, "methods", None) or [])) or "-"
        print(f"{methods:10} {route.path}")

In [ ]:
show_routes(app)

Шість маршрутів агрегатора — `/health`, три `GET` новин, `POST /api/scrape`, `DELETE /api/news` — і чотири службові. У службових маршрутах поруч з `GET` дозволено ще й `HEAD` (той самий запит, але без тіла відповіді). Зверни увагу: `GET /api/news` і `DELETE /api/news` — **один шлях, два методи**, дві різні функції.

### Приклад 1.2. Порожнє сховище

Новин ще ніхто не збирав. Що бачить клієнт?

In [ ]:
print(client.get("/api/news").json(), client.get("/api/news/count").json())

`[]` і `{'count': 0}`: порожній список — це **успіх** (`200`), а не помилка. «Немає даних» і «помилка» — різні відповіді.

### Приклад 1.3. Зібрати новини зі знімка

`POST /api/scrape` з тілом `{"source": "snapshot"}` бере новини зі збереженого знімка, перевіряє кожну моделлю `NewsItem` і кладе нові в сховище. Звіт про збір — у відповіді.

In [ ]:
report = client.post("/api/scrape", json={"source": "snapshot"}).json()
print({key: report[key] for key in ("news_found", "news_valid", "news_saved", "news_total")})
print("відхилено:", report["rejected"])
assert report["news_saved"] == 168
assert client.get("/api/news/count").json() == {"count": 168}

Знайдено 168, усі пройшли перевірку, усі 168 — нові. Список `rejected` порожній: у знімку немає зламаних новин.

**🔮 Прогноз:** Що покаже `news_saved`, якщо зібрати знімок ще раз?

<details>
<summary>Відповідь</summary>

`0`: ключ сховища — `url`, а всі 168 новин уже є. `news_found` і `news_valid` — знову 168, `news_total` — 168.

</details>

### Приклад 1.4. Повторний збір не дублює новин

In [ ]:
again = client.post("/api/scrape", json={"source": "snapshot"}).json()
print({key: again[key] for key in ("news_found", "news_valid", "news_saved", "news_total")})

`news_saved: 0`, `news_total: 168` — повторний запит **безпечний**: виконати його двічі — те саме, що один раз. Це правило `NewsStore.add_many` (url → новина), а не ендпоінта.

### Приклад 1.5. Фільтр і відповідь цілком

`GET /api/news?lang=uk&limit=2` — дві новини з українського розділу. Подивимось не лише JSON, а й статус і заголовок `content-type`.

In [ ]:
response = client.get("/api/news", params={"lang": "uk", "limit": 2})
print(response.status_code, response.headers["content-type"])
for news in response.json():
    print(news)

Два спостереження:

- `published_time` — `None` (у JSON — `null`): у цих новин у стрічці не було часу. Де час був, тут буде рядок `"HH:MM:SS"` — `response_model` сам перетворює `time` на JSON;
- обидві новини з `/ukr/`, тож `lang="uk"`, а заголовки — російською. Мову агрегатор **виводить з URL**, а сайт не завжди кладе текст туди, куди обіцяє адреса. Визначати мову за змістом навчимося в уроці 44.

### Приклад 1.6. Усі новини російською

`limit` за замовчуванням — 50. Щоб отримати **усі** новини `ru`, його треба збільшити (максимум — 1000).

In [ ]:
ru_news = client.get("/api/news", params={"lang": "ru", "limit": 1000}).json()
print(len(ru_news), "|", ru_news[0]["title"])
assert len(ru_news) == 138
assert all(news["lang"] == "ru" for news in ru_news)

138 новин. Без `limit=1000` прийшло б лише 50 — і висновок «російських новин 50» був би хибним. Правило: перевіряй, чи відповідь не обрізана пагінацією.

### Приклад 1.7. Пагінація: `skip` і `limit`

Стрічку віддають сторінками: перша — `skip=0, limit=3`, друга — `skip=3, limit=3`. Функція `page(n)` збирає номер сторінки в параметри.

In [ ]:
def page(number: int, size: int = 3) -> list[str]:
    """Заголовки сторінки number (з 1) розміром size."""
    params = {"skip": (number - 1) * size, "limit": size}
    return [news["title"][:45] for news in client.get("/api/news", params=params).json()]

In [ ]:
first, second = page(1), page(2)
print("1:", first)
print("2:", second)
assert not set(first) & set(second), "сторінки не мають перетинатися"

Сторінки не перетинаються: друга починається там, де закінчилась перша. Формула `skip = (номер − 1) × розмір` — та сама, що `OFFSET` в SQL (урок 39 зробить саме так).

### Приклад 1.8. Статистика

`GET /api/news/stats` — скільки новин у кожній категорії, мові й джерелі. Відповідь описує модель `Stats` (`response_model=Stats`).

In [ ]:
stats = client.get("/api/news/stats").json()
for field, counts in stats.items():
    print(f"{field:9}", counts)

`total` — число, решта — словники «значення → кількість»: 138 + 30 = 168, `rbc.ua` 167 + `auto.rbc.ua` 1 = 168. Категорія у всіх — «Новини», бо знімок зібрано зі стрічки новин.

---
## 2. `422`: помилка перевірки

`limit: int = Query(50, ge=1, le=1000)` — FastAPI перевіряє параметр **до** виклику нашої функції. Неправильне значення до неї навіть не доходить: відповідь `422` і список `detail`, у кожному елементі — `loc` (де), `msg` (що), `input` (що прийшло).

### Приклад 2.1. Завеликий `limit`

In [ ]:
response = client.get("/api/news", params={"limit": 5000})
error = response.json()["detail"][0]
print(response.status_code, error["loc"], "|", error["msg"], "| прийшло:", error["input"])
assert response.status_code == 422 and error["loc"] == ["query", "limit"]

`["query", "limit"]` — параметр **рядка запиту** `limit`; `msg` каже, яке правило порушене (`le=1000`); `input` — те, що прийшло в адресі (`5000`).

### Приклад 2.2. Помічник `show_errors`

Щоб не повторювати розбір `detail`, напишемо функцію, яка друкує всі помилки відповіді одним рядком кожна.

In [ ]:
def show_errors(response: httpx.Response) -> None:
    """Статус і всі помилки перевірки: loc → msg."""
    print(response.status_code)
    for error in response.json()["detail"]:
        print("   ", error["loc"], "→", error["msg"])

In [ ]:
for params in ({"limit": 0}, {"limit": "abc"}, {"lang": "en"}, {"limit": "abc", "lang": "en"}):
    print(params, end=" → ")
    show_errors(client.get("/api/news", params=params))

Кожне правило дає свою помилку: `ge=1` — «greater than or equal to 1», `int` — «valid integer», `Literal["uk", "ru"]` — «'uk' or 'ru'». В останньому запиті **дві** помилки одразу: FastAPI перевіряє всі параметри, а не зупиняється на першому. Клієнт виправляє все за раз.

### Приклад 2.3. Помилки в тілі запиту

Для тіла `loc` починається з `"body"`. `ScrapeRequest` має `mode: Literal["async", "sequential"]` і валідатор, який пропускає лише сторінки rbc.ua.

In [ ]:
show_errors(client.post("/api/scrape", json={"mode": "fast"}))
show_errors(client.post("/api/scrape", json={"pages": ["https://example.com/admin"]}))

- `["body", "mode"]` — поле `mode` у JSON-тілі;
- `["body", "pages"]` і «Value error, …» — це вже **наш** валідатор `only_rbc`: його `ValueError` FastAPI теж перетворює на `422`.

### Приклад 2.4. Тіла немає зовсім або це не JSON

In [ ]:
show_errors(client.post("/api/scrape"))
show_errors(client.post("/api/scrape", content="source=snapshot", headers={"content-type": "application/json"}))

- без тіла — `["body"]`, «Field required»: модель `ScrapeRequest` в аргументах означає «тіло обов'язкове», навіть якщо всі її поля мають значення за замовчуванням (тоді досить `{}`);
- текст замість JSON — «JSON decode error»: FastAPI навіть не дійшов до моделі.

Правило: **`422` — помилка клієнта в даних**. Читай `loc`: перше слово — де шукати (`query`, `path`, `body`), далі — яке поле.

---
## 3. Свої ендпоінти

Невеликий застосунок `mini` — окремо від агрегатора, щоб експериментувати і нічого в ньому не зламати.

### Приклад 3.1. Перший маршрут

`@mini.get("/hello")` — «на `GET /hello` виклич цю функцію». Що функція поверне (словник, список, модель) — FastAPI перетворить на JSON.

In [ ]:
mini = FastAPI(title="mini")


@mini.get("/hello")
async def hello() -> dict[str, str]:
    return {"message": "Привіт, агрегаторе!"}

In [ ]:
mini_client = TestClient(mini)
response = mini_client.get("/hello")
print(response.status_code, response.json())
print(mini_client.get("/bye").status_code, mini_client.get("/bye").json())

`/hello` — `200`; маршруту `/bye` немає — `404 {'detail': 'Not Found'}`, і цю відповідь теж зробив FastAPI.

### Приклад 3.2. Параметр шляху

`{n}` у шляху — **параметр шляху**: частина адреси стає аргументом функції. Тип `int` і `Path(ge=0)` — правила перевірки.

In [ ]:
@mini.get("/square/{n}")
async def square(n: int = Path(ge=0)) -> dict[str, int]:
    return {"n": n, "square": n * n}

In [ ]:
print(mini_client.get("/square/12").json())
for bad in ("/square/-1", "/square/abc"):
    response = mini_client.get(bad)
    print(bad, response.status_code, response.json()["detail"][0]["loc"])

`12` з адреси став числом `12`. `-1` порушує `ge=0`, `abc` — не число: обидва — `422` з `loc == ["path", "n"]`.

**🔮 Прогноз:** Чим параметр шляху (`/square/12`) відрізняється від параметра запиту (`/api/news?limit=12`)? Коли що обирати?

<details>
<summary>Відповідь</summary>

Шлях визначає **ресурс** (яку саме річ: новину, користувача, число) — без нього адреси не існує. Параметр запиту **уточнює** вибірку: фільтр, сортування, сторінка — і зазвичай має значення за замовчуванням (урок 33).

</details>

### Приклад 3.3. Параметри рядка запиту

Аргумент простого типу, якого **немає** в шляху, FastAPI бере з рядка запиту `?word=…&times=…`. Значення за замовчуванням робить параметр необов'язковим; `Query(...)` додає обмеження.

In [ ]:
@mini.get("/repeat")
async def repeat(
    word: str = Query(min_length=1, max_length=20),
    times: int = Query(2, ge=1, le=5),
    sep: Literal[" ", ", ", " | "] = " ",
) -> dict[str, str]:
    return {"result": sep.join([word] * times)}

In [ ]:
print(mini_client.get("/repeat", params={"word": "новини"}).json())
print(mini_client.get("/repeat", params={"word": "новини", "times": 3, "sep": " | "}).json())
show_errors(mini_client.get("/repeat"))
show_errors(mini_client.get("/repeat", params={"word": "x", "times": 9}))

- `word` без значення за замовчуванням — **обов'язковий**: без нього `["query", "word"]` «Field required»;
- `times` за замовчуванням 2, але не більше 5;
- `sep` — лише одне з трьох значень (`Literal`).

### Приклад 3.4. Тіло запиту — Pydantic-модель

Аргумент-модель FastAPI бере з **JSON-тіла**. `Headline`: `title` — 10–300 символів, `lang` — `"uk"` або `"ru"`.

In [ ]:
class Headline(BaseModel):
    title: str = Field(min_length=10, max_length=300)
    lang: Literal["uk", "ru"]


@mini.post("/headline")
async def headline(item: Headline) -> dict[str, int | str]:
    return {"words": len(item.title.split()), "lang": item.lang}

In [ ]:
ok = mini_client.post("/headline", json={"title": "Уряд затвердив новий бюджет", "lang": "uk"})
print(ok.status_code, ok.json())
show_errors(mini_client.post("/headline", json={"title": "Коротко", "lang": "en"}))
assert ok.json() == {"words": 4, "lang": "uk"}

Усередині функції `item` — уже перевірений об'єкт `Headline`: `item.title` точно рядок потрібної довжини. Обидві помилки прийшли разом: `["body", "title"]` і `["body", "lang"]`.

Підсумок: **звідки FastAPI бере параметр** — `{...}` у шляху → шлях; простий тип → рядок запиту; Pydantic-модель → тіло; `Depends(...)` → залежність (розділ 4).

### Приклад 3.5. `response_model`: що саме піде клієнту

Функція може повернути більше, ніж треба показати: внутрішні поля, пароль, службові позначки. `response_model` — **фільтр на виході**: у JSON потрапляють лише поля моделі.

In [ ]:
class PublicNews(BaseModel):
    title: str
    lang: str


@mini.get("/raw-news", response_model=PublicNews)
async def raw_news() -> dict[str, str]:
    return {"title": "Курс гривні стабільний", "lang": "uk", "editor_note": "не публікувати до 10:00"}

In [ ]:
print(mini_client.get("/raw-news").json())

`editor_note` у відповідь **не потрапила**: FastAPI пропустив результат функції через `PublicNews`. Так агрегатор віддає `list[NewsItem]` — рівно ті поля, що в моделі.

### Приклад 3.6. `201 Created` і `404 Not Found`

Невелике сховище закладок у словнику. Два нових інструменти:

- `status_code=201` у декораторі — статус успіху для створення (урок 33);
- `raise HTTPException(404, ...)` — перервати функцію і віддати помилку з поясненням у `detail`.

In [ ]:
BOOKMARKS: dict[int, str] = {}


class Bookmark(BaseModel):
    title: str = Field(min_length=3)


@mini.post("/bookmarks", status_code=201)
async def create_bookmark(item: Bookmark) -> dict[str, int | str]:
    bookmark_id = len(BOOKMARKS) + 1
    BOOKMARKS[bookmark_id] = item.title
    return {"id": bookmark_id, "title": item.title}


@mini.get("/bookmarks/{bookmark_id}")
async def get_bookmark(bookmark_id: int) -> dict[str, int | str]:
    if bookmark_id not in BOOKMARKS:
        raise HTTPException(status_code=404, detail=f"закладки {bookmark_id} немає")
    return {"id": bookmark_id, "title": BOOKMARKS[bookmark_id]}

In [ ]:
created = mini_client.post("/bookmarks", json={"title": "Прочитати про НБУ"})
print(created.status_code, created.json())
print(mini_client.get("/bookmarks/1").status_code, mini_client.get("/bookmarks/1").json())
print(mini_client.get("/bookmarks/7").status_code, mini_client.get("/bookmarks/7").json())

`201` — створено; `200` — знайдено; `404` з нашим текстом — немає. `422` і `404` різні: `422` — «запит неправильний», `404` — «запит правильний, але такої речі немає».

### Приклад 3.7. Порядок маршрутів має значення

FastAPI перевіряє маршрути **в порядку оголошення** і бере перший, що підійшов. Що буде, якщо `/items/{item_id}` оголосити раніше за `/items/latest`?

In [ ]:
order = FastAPI()


@order.get("/items/{item_id}")
async def item_by_id(item_id: str) -> dict[str, str]:
    return {"route": "за id", "item_id": item_id}


@order.get("/items/latest")
async def latest_item() -> dict[str, str]:
    return {"route": "найновіший"}

In [ ]:
order_client = TestClient(order)
print(order_client.get("/items/42").json())
print(order_client.get("/items/latest").json())

`/items/latest` потрапив у **перший** маршрут з `item_id="latest"` — до другої функції запит не дійшов. Якби `item_id` був `int`, клієнт отримав би `422` («latest» — не число), і помилку було б ще важче зрозуміти. Правило: **конкретні шляхи оголошуй перед шляхами з параметрами**.

### Приклад 3.8. Ендпоінт агрегатора: `GET /api/news/sources`

Тепер додамо маршрут у сам агрегатор (розібраний приклад з книги):

1. **Шлях і метод.** Читаємо — `GET`; ресурс — джерела новин → `/api/news/sources`.
2. **Звідки дані.** Сховище — через `StoreDep` (розділ 4); `store.stats()["source"]` уже рахує джерела.
3. **Що повертаємо.** `dict[str, int]` — анотація результату одразу стає схемою в `/docs`.

In [ ]:
@app.get("/api/news/sources", tags=["news"])
async def news_sources(store: StoreDep) -> dict[str, int]:
    """Джерела новин і скільки новин з кожного."""
    return store.stats()["source"]

In [ ]:
sources = client.get("/api/news/sources").json()
print(sources)
assert sources == {"rbc.ua": 167, "auto.rbc.ua": 1}

Маршрут з'явився в застосунку, який уже працює, — і клієнт одразу ним користується. Нової логіки немає: ендпоінт лише **з'єднує** HTTP і метод сховища. Чим тонший ендпоінт, тим легше замінити сховище на базу в уроці 39.

### Приклад 3.9. `GET /api/news/sources` з порогом

Варіант з книги «Зміни приклад»: параметр `min_count` — повертати лише джерела, де новин не менше `min_count`. Окремий шлях, щоб не перевизначати попередній маршрут.

In [ ]:
@app.get("/api/news/sources/popular", tags=["news"])
async def popular_sources(store: StoreDep, min_count: int = Query(1, ge=1)) -> dict[str, int]:
    return {source: count for source, count in store.stats()["source"].items() if count >= min_count}

In [ ]:
print(client.get("/api/news/sources/popular", params={"min_count": 2}).json())
show_errors(client.get("/api/news/sources/popular", params={"min_count": 0}))

`auto.rbc.ua` з однією новиною відсіявся; `min_count=0` — `422`, як і очікує `ge=1`.

### Задача 1. Новина за id: `200`, `404`, `422`

У застосунку `mini` є список `MINI_NEWS`. Додай маршрут `GET /mini-news/{news_id}`:

- `news_id` — ціле, не менше 1 (`Path(ge=1)`); номер — це позиція в списку, починаючи з 1;
- є така новина — повернути словник `{"id": news_id, "title": ..., "lang": ...}`;
- немає — `HTTPException` зі статусом `404` і `detail` `"новини {news_id} немає"`;
- параметр `?upper=true` (тип `bool`, за замовчуванням `False`) — заголовок великими літерами.

Перевірка — у наступній клітинці.

In [ ]:
MINI_NEWS = [
    {"title": "НБУ залишив облікову ставку без змін", "lang": "uk"},
    {"title": "Гривня зміцнилася до долара на міжбанку", "lang": "uk"},
    {"title": "В Киеве открыли новую станцию метро", "lang": "ru"},
]

# YOUR CODE HERE


In [ ]:
first = mini_client.get("/mini-news/1")
assert first.status_code == 200, f"GET /mini-news/1 → {first.status_code}: маршрут не знайдено або він падає"
assert first.json() == {"id": 1, "title": MINI_NEWS[0]["title"], "lang": "uk"}, first.json()
assert mini_client.get("/mini-news/3", params={"upper": "true"}).json()["title"] == MINI_NEWS[2]["title"].upper()
missing = mini_client.get("/mini-news/4")
assert missing.status_code == 404 and missing.json() == {"detail": "новини 4 немає"}, missing.json()
assert mini_client.get("/mini-news/0").status_code == 422, "news_id=0 має дати 422: Path(ge=1)"
assert mini_client.get("/mini-news/abc").status_code == 422
print(first.json(), "|", missing.json())
print("✅ Задача 1 розв'язана")

---
## 4. `Depends` і підміна залежності

`Depends(функція)` — «перед викликом ендпоінта виклич цю функцію і передай результат». Ендпоінт оголошує, **що** йому потрібно, а не **звідки** це взяти. Агрегатор так отримує сховище (`StoreDep`) і скрапер (`Depends(get_scrapers)`).

### Приклад 4.1. Спільні параметри пагінації

Два ендпоінти з однаковими `skip`/`limit`. Замість повторювати `Query(...)` двічі — одна залежність `pagination`, яка сама бере параметри з рядка запиту.

In [ ]:
deps = FastAPI()
TITLES = [f"Новина №{number}" for number in range(1, 21)]


def pagination(skip: int = Query(0, ge=0), limit: int = Query(5, ge=1, le=10)) -> dict[str, int]:
    return {"skip": skip, "limit": limit}


PageDep = Annotated[dict[str, int], Depends(pagination)]


@deps.get("/titles")
async def titles(page: PageDep) -> list[str]:
    return TITLES[page["skip"]:page["skip"] + page["limit"]]


@deps.get("/titles/reversed")
async def titles_reversed(page: PageDep) -> list[str]:
    reversed_titles = TITLES[::-1]
    return reversed_titles[page["skip"]:page["skip"] + page["limit"]]

In [ ]:
deps_client = TestClient(deps)
print(deps_client.get("/titles", params={"skip": 2, "limit": 3}).json())
print(deps_client.get("/titles/reversed").json())
show_errors(deps_client.get("/titles/reversed", params={"limit": 50}))

Обидва ендпоінти отримали готовий словник `page`; правила `ge`/`le` описані **один раз** — і перевіряються в обох. `Annotated[тип, Depends(...)]` — тип для mypy і `/docs` плюс інструкція для FastAPI; `PageDep` — щоб не повторювати запис.

### Приклад 4.2. Підміна залежності: `dependency_overrides`

`app.dependency_overrides[стара] = нова` — FastAPI викликатиме `нова` замість `стара` в **усіх** ендпоінтах. Так у тестах підставляють заглушки замість бази чи мережі.

In [ ]:
def small_pages() -> dict[str, int]:
    return {"skip": 0, "limit": 2}

In [ ]:
deps.dependency_overrides[pagination] = small_pages
print("з підміною:", deps_client.get("/titles", params={"limit": 10}).json())
deps.dependency_overrides.clear()
print("без підміни:", deps_client.get("/titles", params={"limit": 10}).json()[:3], "…")

З підміною `limit=10` з адреси **ігнорується**: параметри брала `pagination`, а її замінили. Після `clear()` усе як було. Тому після підміни завжди прибирай за собою — інакше наступний тест працюватиме з заглушкою.

### Приклад 4.3. `lifespan`: код при старті й зупинці

Агрегатор створює сховище в `lifespan`: код до `yield` — один раз при старті, після `yield` — при зупинці. Подивимось порядок на маленькому застосунку.

In [ ]:
events: list[str] = []


@asynccontextmanager
async def demo_lifespan(application: FastAPI) -> AsyncIterator[None]:
    events.append("старт: відкрили сховище")
    application.state.counter = 0
    yield
    events.append(f"стоп: оброблено запитів — {application.state.counter}")


life = FastAPI(lifespan=demo_lifespan)


@life.get("/ping")
async def ping() -> dict[str, int]:
    life.state.counter += 1
    events.append(f"запит {life.state.counter}")
    return {"count": life.state.counter}

In [ ]:
with TestClient(life) as life_client:
    life_client.get("/ping")
    life_client.get("/ping")
for event in events:
    print(event)

Старт — до першого запиту, стоп — після виходу з `with`. Саме тому агрегатор після перезапуску (`__exit__` + `__enter__`) знову порожній: `NewsStore` створюється в `lifespan` заново. Урок 39 замінить це базою.

### Приклад 4.4. Як агрегатор бере скрапер

`POST /api/scrape` без `source="snapshot"` іде в мережу: `scrapers = Depends(get_scrapers)` повертає `{"async": ..., "sequential": ...}`. Справжній `get_scrapers` не викликаємо (мережі немає), а лише дивимось, що він дає:

In [ ]:
for mode, scraper in get_scrapers().items():
    print(f"{mode:10} → {scraper.__module__}.{scraper.__name__}")

Дві асинхронні функції з `scraper.py`. Обидві мають підпис `async (pages) -> ScrapeOutcome`. Будь-яка функція з таким підписом підійде як заміна — це і є Задача 2.

### Задача 2. Заглушка скрапера

Напиши `fake_scraper(pages)` — асинхронну функцію, яка **без мережі** повертає `ScrapeOutcome`:

- `pages` — одна сторінка `PageResult(url="https://www.rbc.ua/ukr/news/", start=0.0, end=0.1, count=2)`;
- `news` — дві сирі новини: `good` і `short` (у `short` заголовок коротший за 10 символів — `NewsItem` його відхилить);
- `total_time=0.1`.

Підміну залежності й запит робить клітинка перевірки.

In [ ]:
good: RawNews = {"title": "Гривня зміцнилася до долара на міжбанку", "url": "https://www.rbc.ua/ukr/news/hryvnia-777.html",
                 "category": "", "description": "", "datetime": "09:05"}
short: RawNews = {"title": "Коротко", "url": "https://www.rbc.ua/ukr/news/short-778.html",
                  "category": "", "description": "", "datetime": ""}


async def fake_scraper(pages: list[str] | None) -> ScrapeOutcome:
    # YOUR CODE HERE
    pass


In [ ]:
app.dependency_overrides[get_scrapers] = lambda: {"async": fake_scraper, "sequential": fake_scraper}
try:
    response = TestClient(app, raise_server_exceptions=False).post("/api/scrape", json={"mode": "sequential"})
finally:
    app.dependency_overrides.clear()

assert response.status_code == 200, f"POST /api/scrape → {response.status_code}: fake_scraper має повернути ScrapeOutcome"
live = response.json()
assert (live["news_found"], live["news_valid"], live["news_saved"]) == (2, 1, 1), live
assert live["rejected"][0].startswith(short["url"])
print({key: live[key] for key in ("news_found", "news_valid", "news_saved", "news_total")})
print(live["rejected"])
print("✅ Задача 2 розв'язана")

Зверни увагу на звіт: `news_found: 2`, `news_valid: 1` — коротку новину відхилив `NewsItem`, і причина видна в `rejected`. Ендпоінт не знав, що скрапер підроблений: він отримав «щось із потрібним підписом» через `Depends`.

---
## 5. OpenAPI

`GET /openapi.json` — опис API за стандартом OpenAPI, який FastAPI будує сам: з маршрутів, типів, моделей і docstring-ів. З нього зроблені `/docs` (Swagger UI) і `/redoc`, його імпортує Postman.

### Приклад 5.1. Що в описі

In [ ]:
schema = client.get("/openapi.json").json()
print(schema["openapi"], "|", schema["info"]["title"], schema["info"]["version"])
for path, methods in schema["paths"].items():
    for method, operation in methods.items():
        print(f"{method.upper():6} {path:28} {operation['tags']} {operation['summary']}")

Кожна операція — метод, шлях, теги, короткий опис. Де ми написали `summary="Список новин"` — він і показаний; де ні — FastAPI зробив його з назви функції (`news_count` → «News Count»). Маршрути з розділу 3 (`/api/news/sources…`) теж уже тут: опис оновлюється разом з кодом.

### Приклад 5.2. Шляхи з тегом `news`

Теги групують ендпоінти в `/docs`. Зберемо множину шляхів, у яких **хоч одна** операція має тег `"news"`.

In [ ]:
news_paths = {path for path, methods in schema["paths"].items()
              if any("news" in operation.get("tags", []) for operation in methods.values())}
print(sorted(news_paths))
assert {"/api/news", "/api/news/count", "/api/news/stats", "/api/news/sources"} <= news_paths

`any(...)` — бо в одного шляху кілька операцій (`GET` і `DELETE` для `/api/news`). `/api/scrape` сюди не потрапив: його тег — `scrape`.

### Приклад 5.3. Схема моделі `NewsItem`

`components.schemas` — моделі. Опис `NewsItem` — та сама JSON Schema, що в уроці 37 дав `NewsItem.model_json_schema()`.

In [ ]:
print(sorted(schema["components"]["schemas"]))
news_item = schema["components"]["schemas"]["NewsItem"]
print("обов'язкові:", news_item["required"])
for name, field in news_item["properties"].items():
    print(f"   {name:15}", {key: value for key, value in field.items() if key != "title"})

Обмеження `min_length=10`, `format: uri`, `enum` для `lang` — усе з моделі, і саме за цим описом Swagger UI підказує, що вводити. `published_time` не в `required`: має значення за замовчуванням `None`.

---
## 6. `async def` чи `def`

FastAPI приймає обидва варіанти. Різниця — **хто** виконує функцію:

- `async def` — виконується в **циклі подій** (один потік на всі запити); поки корутина чекає на `await`, цикл обслуговує інших;
- `def` — FastAPI запускає в **пулі потоків**, щоб блокуючий код не зупиняв цикл.

Небезпечно — `async def` + блокуючий виклик (`time.sleep`, `requests.get`, синхронний драйвер бази): функція займає цикл, і чекає **весь сервер**.

### Приклад 6.1. Лабораторія: три ендпоінти й вимірювач

`httpx.AsyncClient` з `ASGITransport` надсилає запити в застосунок **одночасно** (`asyncio.gather`) — як багато клієнтів до сервера.

In [ ]:
lab = FastAPI()


@lab.get("/blocking")
async def blocking() -> dict[str, str]:
    time.sleep(0.3)                 # займає цикл подій
    return {"kind": "blocking"}


@lab.get("/awaiting")
async def awaiting() -> dict[str, str]:
    await asyncio.sleep(0.3)        # віддає цикл подій іншим
    return {"kind": "awaiting"}


@lab.get("/threaded")
def threaded() -> dict[str, str]:
    time.sleep(0.3)                 # def → FastAPI виконає в пулі потоків
    return {"kind": "threaded"}


async def measure(path: str, n: int = 5) -> float:
    """n одночасних запитів до path; загальний час у секундах."""
    transport = httpx.ASGITransport(app=lab)
    async with httpx.AsyncClient(transport=transport, base_url="http://lab") as http:
        start = time.perf_counter()
        await asyncio.gather(*(http.get(path) for _ in range(n)))
        return round(time.perf_counter() - start, 2)

**🔮 Прогноз:** 5 одночасних запитів, кожен «чекає» 0,3 с. Скільки триватимуть разом `/blocking`, `/awaiting` і `/threaded`?

<details>
<summary>Відповідь</summary>

`/blocking` — ~1,5 с (5 × 0,3: `time.sleep` тримає цикл подій, запити йдуть по черзі). `/awaiting` — ~0,3 с (усі чекають одночасно). `/threaded` — теж ~0,3 с: кожен запит спить у своєму потоці пулу. На сервері з книги це 40 с проти 2 с для 20 запитів по 2 с.

</details>

### Приклад 6.2. Вимір

In [ ]:
t_blocking = await measure("/blocking")
t_awaiting = await measure("/awaiting")
t_threaded = await measure("/threaded")
print(f"blocking {t_blocking} с | awaiting {t_awaiting} с | threaded {t_threaded} с")
assert t_blocking >= 1.4
assert t_awaiting < 0.8 and t_threaded < 0.8

Числа в тебе трохи інші, а співвідношення — те саме: `blocking` у ~5 разів довший. У Jupyter можна писати `await` прямо в клітинці — тут уже працює цикл подій.

### Приклад 6.3. Блокування зачіпає **інших**

Найгірше в `/blocking` не те, що він повільний, а те, що поки він спить, не відповідає **ніхто**. Надішлемо три `/blocking` і одночасно швидкий `/fast` — скільки чекатиме `/fast`?

In [ ]:
@lab.get("/fast")
async def fast() -> dict[str, str]:
    return {"kind": "fast"}


async def fast_latency(slow_path: str) -> float:
    """Через скільки секунд від старту готовий /fast, якщо разом з ним пішли три запити до slow_path."""
    transport = httpx.ASGITransport(app=lab)
    async with httpx.AsyncClient(transport=transport, base_url="http://lab") as http:
        start = time.perf_counter()

        async def timed_fast() -> float:
            await http.get("/fast")
            return time.perf_counter() - start

        *_, latency = await asyncio.gather(*(http.get(slow_path) for _ in range(3)), timed_fast())
        return round(latency, 2)

In [ ]:
print("/fast поруч з /blocking:", await fast_latency("/blocking"), "с")
print("/fast поруч з /awaiting:", await fast_latency("/awaiting"), "с")

Поруч з `/blocking` миттєвий `/fast` готовий лише через ~0,9 с — поки по черзі «проспляться» три заблоковані запити (3 × 0,3 с). Поруч з `/awaiting` — одразу: ті три запити чекають на `await` і цикл подій вільний. У книзі так само `/health` шість разів поспіль не відповів за 2 с: сервер з `time.sleep` в `async def` не відповідає **нікому**.

### Приклад 6.4. Блокуючий виклик, від якого не втекти: `asyncio.to_thread`

Буває, що функція мусить бути `async def` (усередині є `await`), але потрібна й синхронна бібліотека. Тоді блокуючий виклик виносимо в потік: `await asyncio.to_thread(функція, аргументи)` — цикл подій вільний, поки потік працює.

In [ ]:
def slow_sync_driver(news_id: int) -> dict[str, int]:
    """Імітація синхронного драйвера бази: 0,3 с блокуючого очікування."""
    time.sleep(0.3)
    return {"news_id": news_id}


@lab.get("/db-broken/{news_id}")
async def db_broken(news_id: int) -> dict[str, int]:
    return slow_sync_driver(news_id)                          # блокує цикл


@lab.get("/db-fixed/{news_id}")
async def db_fixed(news_id: int) -> dict[str, int]:
    return await asyncio.to_thread(slow_sync_driver, news_id)  # у потоці, цикл вільний

In [ ]:
print("db-broken:", await measure("/db-broken/1"), "с")
print("db-fixed: ", await measure("/db-fixed/1"), "с")

Той самий драйвер: «зламаний» варіант — ~1,5 с на 5 запитів, з `to_thread` — ~0,3 с. Це `/db-sync-broken` і `/db-async-correct` з `fastapi_demo` у книзі.

| Що всередині ендпоінта | Як писати |
|---|---|
| лише `await` (aiohttp, `asyncio.sleep`, async-драйвер бази) | `async def` |
| лише блокуючий код (`requests`, `time.sleep`, синхронний драйвер) | `def` |
| і те, і те | `async def` + `await asyncio.to_thread(...)` для блокуючого |

---
## 7. Знайди помилку: чужий URL

Студент додав ендпоінт «зібрати одну сторінку»:

```python
@app.post("/api/scrape/page")
async def scrape_page(url: str, store: StoreDep) -> dict[str, int]:
    html = requests.get(url, timeout=15).text
    ...
```

Дві проблеми:

1. `requests.get` у `async def` блокує цикл подій — це `/blocking` з розділу 6;
2. `url: str` — будь-яка адреса, зокрема внутрішня (`http://localhost:…`, адреса службової інформації хмари `169.254.169.254`): сервер завантажить її на прохання будь-кого. Це **SSRF** (Server-Side Request Forgery), детально — урок 47.

### Приклад 7.1. Що `HttpUrl` знає про адресу

Перш ніж перевіряти домен, подивимось, на що Pydantic розбирає URL. `.host` — лише ім'я хоста, без схеми, порту й шляху.

In [ ]:
class AnyPage(BaseModel):
    url: HttpUrl

In [ ]:
for raw in ("https://www.rbc.ua/ukr/news/", "https://auto.rbc.ua/rus/news/x.html",
            "http://localhost:8000/api/news", "https://rbc.ua.evil.com/", "https://fakerbc.ua/news"):
    page = AnyPage(url=raw)
    print(f"{raw:40} host={page.url.host!r:20} port={page.url.port}")
try:
    AnyPage(url="не URL")
except ValidationError as error:
    print("не URL →", error.errors()[0]["msg"])

Дві пастки для перевірки «чи це rbc.ua»:

- `"rbc.ua" in url` пропустить `https://rbc.ua.evil.com/` — у рядку є `rbc.ua`, але хост — `evil.com`;
- `host.endswith("rbc.ua")` пропустить `fakerbc.ua` — зовсім інший домен.

Правильна перевірка: хост **дорівнює** `rbc.ua` або закінчується на **`.rbc.ua`** (з крапкою — тобто піддомен).

### Приклад 7.2. Як `ScrapeRequest` агрегатора реагує на чужі адреси

In [ ]:
for pages in (["https://www.rbc.ua/ukr/news/"], ["http://localhost:8000/api/news"], ["http://169.254.169.254/latest/"]):
    try:
        ScrapeRequest(pages=pages)
        print(pages[0], "→ дозволено")
    except ValidationError as error:
        print(pages[0], "→", error.errors()[0]["msg"])

Валідатор моделі спрацьовує ще **до** ендпоінта: внутрішня адреса не дійде до скрапера. Те саме зробимо для одного URL.

### Задача 3. `PageRequest`: лише rbc.ua

Опиши модель `PageRequest` з полем `url: HttpUrl` і валідатором поля (`@field_validator("url")`):

- дозволено, якщо хост дорівнює `rbc.ua` **або** закінчується на `.rbc.ua`;
- інакше — `ValueError(f"лише rbc.ua, а не {url.host}")`.

Перевірка — у наступній клітинці.

In [ ]:
# YOUR CODE HERE


In [ ]:
assert "PageRequest" in globals(), "спершу опиши клас PageRequest"
assert PageRequest(url="https://auto.rbc.ua/rus/news/x.html").url.host == "auto.rbc.ua"
assert PageRequest(url="https://rbc.ua/").url.host == "rbc.ua"
for bad in ("http://localhost:8000/api/news", "http://169.254.169.254/latest/meta-data/",
            "https://fakerbc.ua/news", "https://rbc.ua.evil.com/", "не URL"):
    try:
        PageRequest(url=bad)
    except ValidationError as error:
        print(f"{bad:42} →", error.errors()[0]["msg"])
    else:
        raise AssertionError(f"{bad} пропущено — перевір умову для хоста")
print("✅ Задача 3 розв'язана")

---
## Самоперевірка

1. Звідки FastAPI бере значення параметра: шлях, рядок запиту, тіло?
2. Що означає `422` і де шукати причину? Чим він відрізняється від `404`?
3. Навіщо сховище через `Depends`, а не глобальна змінна?
4. Коли ендпоінт писати як `async def`, а коли як `def`?
5. Що таке SSRF і чому `endswith("rbc.ua")` — недостатня перевірка?

<details>
<summary>Відповіді</summary>

1. `{n}` у шляху — параметр шляху; простий тип без `{}` — рядок запиту (`?limit=`); Pydantic-модель — JSON-тіло; `Depends` — залежність.
2. Дані не пройшли перевірку; `detail[i].loc` — де (`["query", "limit"]`), `msg` — що не так. `404` — запит правильний, але такого ресурсу немає.
3. Залежність підміняється в тестах (`dependency_overrides`) і замінюється іншою реалізацією (SQLAlchemy в уроці 39) без змін в ендпоінтах.
4. `async def` — коли всередині лише `await`. Є блокуючий виклик — `def` (FastAPI виконає в пулі потоків), async-бібліотека або `asyncio.to_thread`.
5. Server-Side Request Forgery: сервер завантажує адресу, яку підсунув клієнт, — зокрема внутрішню. `endswith("rbc.ua")` пропускає `fakerbc.ua`; треба `host == "rbc.ua"` або `host.endswith(".rbc.ua")`.

</details>

## Далі

- Книга курсу: [Урок 38](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_38/) — три рефакторинги, архітектура, `/docs`, Postman, виміри `fastapi_demo`.
- У папці `news_hub`: `uvicorn news_hub.api:app --reload` → http://127.0.0.1:8000/docs; `pytest`; колекція `postman/`.
- **Урок 39** — `NewsStore` → SQLAlchemy: новини в базі.

In [ ]:
client.__exit__(None, None, None)   # зупинити lifespan (як вихід з with)